# Masking thesis: train and score the detector

Runner notebook. **The code lives in GitHub; this notebook only runs it.** Everyone gets their own runtime
(GPU session); only the notebook text is shared, not the running machine. Save results to Drive so they survive disconnects.

Run top to bottom: setup, data, baselines, smoke test, full training.
Runtime menu: **Change runtime type, GPU** before step 4.

In [ ]:
# 1) Setup: Drive for outputs, GitHub for code
from google.colab import drive
import os
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/masking-thesis'   # results and checkpoints live here
os.makedirs(WORK, exist_ok=True)

REPO = 'https://github.com/YOUR-TEAM/masking-thesis.git'   # <- CHANGE THIS
if not os.path.exists('/content/code'):
    !git clone $REPO /content/code
else:
    !git -C /content/code pull
%cd /content/code

In [ ]:
# 2) Dependencies (torch is preinstalled on Colab)
!pip -q install transformers accelerate

In [ ]:
# 3) Data: regenerated deterministically from the seed (no need to store it)
!python generate_dataset.py --out data

In [ ]:
# 4) Baselines first: this checks that the scorer works and gives you the numbers to beat
!python baseline_regex.py --data data/test.jsonl --out preds_regex.jsonl
!python baseline_regex.py --data data/test.jsonl --out preds_regex_cues.jsonl --cues
!python score.py --gold data/test.jsonl --pred preds_regex_cues.jsonl

In [ ]:
# 5) Smoke test: a few minutes. If this fails, fix it before the full run.
!python train_ner.py --data data --model xlm-roberta-base --out {WORK}/runs/smoke --epochs 1 --limit 200

In [ ]:
# 6) Full run (multilingual encoder)
!python train_ner.py --data data --model xlm-roberta-base --out {WORK}/runs/xlmr --epochs 5

In [ ]:
# 7) Comparison run (IndoBERT). Needs a fast tokenizer; the script stops with a message if not.
!python train_ner.py --data data --model indobenchmark/indobert-base-p1 --out {WORK}/runs/indobert --epochs 5

## Habits that save you pain
- Edit `.py` files in GitHub (branches or pull requests), not inside the notebook. Notebooks merge badly.
- Never paste API keys into a shared notebook. For the later LLM-evaluation step, use Colab **Secrets** (key icon in the left sidebar) and read them with `from google.colab import userdata`.
- Free GPU sessions can disconnect and are not guaranteed, so checkpoints go to `WORK` on Drive (already set above).
- Commit `report_*.json` files to the repo so everyone compares the same numbers.